# A7 Architecture lead — TinyLlama on Colab GPU

Select a T4 GPU runtime and run the cells in order. No Hugging Face token is needed for this public model. The shared 25 prompts, greedy generation and 220-token limit are unchanged. Sources are embedded with SHA256 checks. Keep these CUDA/float16 timings separate from Windows CPU/float32 timings. Llama-3.2 remains subject to the author’s gated access approval.

In [ ]:
import sys, subprocess, os, json, hashlib
from pathlib import Path
WORK = Path('/content/hestia_a7_architecture')
WORK.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = str(WORK/'model_cache/huggingface')
os.environ['HF_HUB_DISABLE_XET'] = '1'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.6', 'huggingface-hub==0.36.2', 'accelerate==1.15.0', 'psutil>=5.9'])
SOURCES = {'__init__.py': '"""Assignment 7 experiment materials."""\n', 'benchmark_contract.py': '"""Locked model, prompt, and evaluation contract for Assignment 7."""\n\nfrom collections import Counter\n\n\nMODEL_REGISTRY = [\n    {"model_id": "distilbert/distilgpt2", "params_b": 0.082, "category": "Ultra-light (<0.2B)", "owner": "P1"},\n    {"model_id": "facebook/opt-125m", "params_b": 0.125, "category": "Ultra-light (<0.2B)", "owner": "P2"},\n    {"model_id": "HuggingFaceTB/SmolLM2-135M-Instruct", "params_b": 0.135, "category": "Ultra-light (<0.2B)", "owner": "P3"},\n    {"model_id": "HuggingFaceTB/SmolLM-360M-Instruct", "params_b": 0.360, "category": "Compact (0.2B–<0.6B)", "owner": "architecture"},\n    {"model_id": "HuggingFaceTB/SmolLM2-360M-Instruct", "params_b": 0.360, "category": "Compact (0.2B–<0.6B)", "owner": "P1"},\n    {"model_id": "Qwen/Qwen2-0.5B-Instruct", "params_b": 0.494, "category": "Compact (0.2B–<0.6B)", "owner": "P2"},\n    {"model_id": "Qwen/Qwen3-0.6B", "params_b": 0.600, "category": "Small (0.6B–<1.6B)", "owner": "P3"},\n    {"model_id": "TinyLlama/TinyLlama-1.1B-Chat-v1.0", "params_b": 1.100, "category": "Small (0.6B–<1.6B)", "owner": "architecture"},\n    {"model_id": "Qwen/Qwen2.5-1.5B-Instruct", "params_b": 1.540, "category": "Small (0.6B–<1.6B)", "owner": "P1"},\n    {"model_id": "HuggingFaceTB/SmolLM2-1.7B-Instruct", "params_b": 1.700, "category": "Medium (1.6B–<3.5B)", "owner": "P2"},\n    {"model_id": "Qwen/Qwen2.5-3B-Instruct", "params_b": 3.090, "category": "Medium (1.6B–<3.5B)", "owner": "P3"},\n    {"model_id": "meta-llama/Llama-3.2-3B-Instruct", "params_b": 3.210, "category": "Medium (1.6B–<3.5B)", "owner": "architecture", "access": "gated"},\n    {"model_id": "microsoft/Phi-3.5-mini-instruct", "params_b": 3.820, "category": "Large (≥3.5B)", "owner": "P1"},\n    {"model_id": "mistralai/Mistral-7B-Instruct-v0.3", "params_b": 7.250, "category": "Large (≥3.5B)", "owner": "P2"},\n    {"model_id": "Qwen/Qwen2.5-7B-Instruct", "params_b": 7.620, "category": "Large (≥3.5B)", "owner": "P3"},\n]\n\nfor model in MODEL_REGISTRY:\n    model["model_card"] = f"https://huggingface.co/{model[\'model_id\']}"\n\n\nPROMPTS = [\n    # Structured extraction\n    {"prompt_id": "extraction-01", "prompt_type": "structured_extraction", "input": "Guests arrive tomorrow. Clean the kitchen, buy snacks, and fix the leaking bathroom tap."},\n    {"prompt_id": "extraction-02", "prompt_type": "structured_extraction", "input": "This week I need to book the dentist, replace smoke-detector batteries, return a library book by Friday, and wash the guest sheets."},\n    {"prompt_id": "extraction-03", "prompt_type": "structured_extraction", "input": "Before Saturday, vacuum the living room and pick up the birthday cake. The loose stair rail also needs repair."},\n    {"prompt_id": "extraction-04", "prompt_type": "structured_extraction", "input": "Please remind us to pay the electricity bill tonight, schedule the dog\'s vet visit, and get laundry detergent."},\n    {"prompt_id": "extraction-05", "prompt_type": "structured_extraction", "input": "The refrigerator is making noise. Also organize the hall closet and mail the tax documents before Monday."},\n    # Classification\n    {"prompt_id": "classification-01", "prompt_type": "classification", "input": "Buy milk and rice after work."},\n    {"prompt_id": "classification-02", "prompt_type": "classification", "input": "Call the plumber about the dripping kitchen pipe."},\n    {"prompt_id": "classification-03", "prompt_type": "classification", "input": "Schedule a yearly dental cleaning."},\n    {"prompt_id": "classification-04", "prompt_type": "classification", "input": "Submit the apartment renewal form by Friday."},\n    {"prompt_id": "classification-05", "prompt_type": "classification", "input": "Sweep the porch and empty the recycling."},\n    # Priority prediction\n    {"prompt_id": "priority-01", "prompt_type": "priority_prediction", "input": "The smoke detector is chirping because its battery is low."},\n    {"prompt_id": "priority-02", "prompt_type": "priority_prediction", "input": "Sort the photo drawer whenever there is free time."},\n    {"prompt_id": "priority-03", "prompt_type": "priority_prediction", "input": "Return the library book tomorrow to avoid a fee."},\n    {"prompt_id": "priority-04", "prompt_type": "priority_prediction", "input": "Water is rapidly leaking under the bathroom sink."},\n    {"prompt_id": "priority-05", "prompt_type": "priority_prediction", "input": "Buy coffee sometime this week; enough remains for two days."},\n    # Summarization\n    {"prompt_id": "summary-01", "prompt_type": "summarization", "input": "The pantry shelves have crumbs and expired cans. Remove old food, wipe each shelf, and group the remaining items."},\n    {"prompt_id": "summary-02", "prompt_type": "summarization", "input": "Call the pediatric clinic, ask for an appointment after 3 PM next week, and write the confirmed time on the family calendar."},\n    {"prompt_id": "summary-03", "prompt_type": "summarization", "input": "Compare the electricity statements from the last three months and contact the provider if the unexplained increase continues."},\n    {"prompt_id": "summary-04", "prompt_type": "summarization", "input": "Gather the borrowed books from the bedrooms, check their due dates, and return everything due this weekend."},\n    {"prompt_id": "summary-05", "prompt_type": "summarization", "input": "Before the guests arrive, replace the guest-room sheets, add clean towels, empty the bin, and check the bedside lamp."},\n    # Action planning\n    {"prompt_id": "planning-01", "prompt_type": "action_planning", "input": "Prepare the apartment for overnight guests tomorrow."},\n    {"prompt_id": "planning-02", "prompt_type": "action_planning", "input": "Organize a weekly grocery trip while staying under a $90 budget."},\n    {"prompt_id": "planning-03", "prompt_type": "action_planning", "input": "Deep-clean the kitchen safely in one afternoon."},\n    {"prompt_id": "planning-04", "prompt_type": "action_planning", "input": "Get ready for a move to a new apartment next month."},\n    {"prompt_id": "planning-05", "prompt_type": "action_planning", "input": "Create a simple plan for reducing household electricity use this week."},\n]\n\n\nEVALUATION_CRITERIA = [\n    {"criterion": "task_accuracy", "max_points": 3, "description": "Correctly addresses the household request without changing its meaning."},\n    {"criterion": "instruction_adherence", "max_points": 2, "description": "Follows the requested task and output instructions."},\n    {"criterion": "completeness", "max_points": 2, "description": "Includes all essential requested information."},\n    {"criterion": "clarity", "max_points": 2, "description": "Produces concise, understandable, usable text."},\n    {"criterion": "no_unsupported_details", "max_points": 1, "description": "Avoids invented facts, dates, people, or requirements."},\n]\n\n\nOUTPUT_INSTRUCTIONS = {\n    "structured_extraction": (\n        \'Return JSON only: {"tasks":[{"title":"...","task_type":"CHORE|APPOINTMENT|DEADLINE|ERRAND|MAINTENANCE|SHOPPING|OTHER",\'\n        \'"priority":"LOW|MEDIUM|HIGH|URGENT","description":"..."}]}. Create one task per concern.\'\n    ),\n    "classification": "Return exactly one label: CHORE, APPOINTMENT, DEADLINE, ERRAND, MAINTENANCE, SHOPPING, or OTHER.",\n    "priority_prediction": "Return exactly one label: LOW, MEDIUM, HIGH, or URGENT. Use urgency and safety risk, not dramatic wording.",\n    "summarization": \'Return JSON only: {"title":"short action title","summary":"one concise sentence"}.\',\n    "action_planning": \'Return JSON only: {"steps":["step 1","step 2","step 3"]}. Use 3 to 5 practical ordered steps.\',\n}\n\n\ndef render_prompt(prompt: dict[str, str]) -> str:\n    """Render one fixed benchmark prompt without model-specific changes."""\n    return (\n        "You are Hestia, a household coordination assistant. "\n        "Do not invent private details or claim that an action is complete.\\n"\n        f"{OUTPUT_INSTRUCTIONS[prompt[\'prompt_type\']]}\\n"\n        f"Household request: {prompt[\'input\']}\\n"\n        "Answer:"\n    )\n\n\ndef validate_contract() -> None:\n    """Raise AssertionError if the shared experiment matrix drifts."""\n    assert len(MODEL_REGISTRY) == 15\n    assert len({model["model_id"] for model in MODEL_REGISTRY}) == 15\n    assert set(Counter(model["category"] for model in MODEL_REGISTRY).values()) == {3}\n    assert Counter(model["owner"] for model in MODEL_REGISTRY) == {"P1": 4, "P2": 4, "P3": 4, "architecture": 3}\n    assert len(PROMPTS) == 25\n    assert len({prompt["prompt_id"] for prompt in PROMPTS}) == 25\n    assert set(Counter(prompt["prompt_type"] for prompt in PROMPTS).values()) == {5}\n    assert set(OUTPUT_INSTRUCTIONS) == {prompt["prompt_type"] for prompt in PROMPTS}\n    assert sum(item["max_points"] for item in EVALUATION_CRITERIA) == 10\n\n\nvalidate_contract()\n', 'benchmark_runner.py': '"""Run the shared Assignment 7 prompt matrix against selected local models."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport gc\nimport json\nimport time\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Iterable\n\nfrom a7.benchmark_contract import MODEL_REGISTRY, PROMPTS, render_prompt, validate_contract\n\n\nRESULT_COLUMNS = [\n    "model_id",\n    "model_category",\n    "owner",\n    "prompt_id",\n    "prompt_type",\n    "prompt",\n    "latency_seconds",\n    "output_tokens",\n    "tokens_per_second",\n    "generated_output",\n    "status",\n    "error",\n    "device",\n    "max_new_tokens",\n    "generated_at_utc",\n    "task_accuracy_0_to_3",\n    "instruction_adherence_0_to_2",\n    "completeness_0_to_2",\n    "clarity_0_to_2",\n    "no_unsupported_details_0_to_1",\n    "self_evaluation_score_1_to_10",\n    "review_notes",\n]\n\n\ndef select_models(owner: str | None = None, model_ids: Iterable[str] | None = None) -> list[dict]:\n    """Return roster entries for one owner or an explicit list of model IDs."""\n    known = {model["model_id"]: model for model in MODEL_REGISTRY}\n    if model_ids:\n        requested = list(model_ids)\n        unknown = [model_id for model_id in requested if model_id not in known]\n        if unknown:\n            raise ValueError(f"Unknown model ID(s): {\', \'.join(unknown)}")\n        return [known[model_id] for model_id in requested]\n    if owner:\n        selected = [model for model in MODEL_REGISTRY if model["owner"] == owner]\n        if not selected:\n            raise ValueError(f"Unknown owner: {owner}")\n        return selected\n    return list(MODEL_REGISTRY)\n\n\ndef expected_keys(models: Iterable[dict], prompts: Iterable[dict]) -> list[tuple[str, str]]:\n    return [(model["model_id"], prompt["prompt_id"]) for model in models for prompt in prompts]\n\n\ndef validate_result_rows(rows: list[dict], models: list[dict], prompts: list[dict]) -> None:\n    """Validate row schema and exact model/prompt coverage."""\n    for row in rows:\n        missing = set(RESULT_COLUMNS) - set(row)\n        if missing:\n            raise ValueError(f"Result row missing columns: {sorted(missing)}")\n    keys = [(row["model_id"], row["prompt_id"]) for row in rows]\n    if len(keys) != len(set(keys)):\n        raise ValueError("Duplicate model/prompt result rows found")\n    expected = set(expected_keys(models, prompts))\n    actual = set(keys)\n    if actual != expected:\n        raise ValueError(f"Result coverage mismatch: missing={len(expected-actual)}, extra={len(actual-expected)}")\n\n\ndef _read_rows(path: Path) -> list[dict]:\n    if not path.exists():\n        return []\n    with path.open(newline="", encoding="utf-8") as handle:\n        return list(csv.DictReader(handle))\n\n\ndef _write_rows(path: Path, rows: list[dict]) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=RESULT_COLUMNS)\n        writer.writeheader()\n        writer.writerows(rows)\n    temporary.replace(path)\n\n\ndef _device_name(requested: str, torch) -> str:\n    if requested != "auto":\n        return requested\n    if torch.cuda.is_available():\n        return "cuda"\n    if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():\n        return "mps"\n    return "cpu"\n\n\ndef _synchronize(device: str, torch) -> None:\n    if device == "cuda":\n        torch.cuda.synchronize()\n    elif device == "mps":\n        torch.mps.synchronize()\n\n\ndef _format_input(tokenizer, prompt_text: str) -> str:\n    if getattr(tokenizer, "chat_template", None):\n        messages = [{"role": "user", "content": prompt_text}]\n        try:\n            return tokenizer.apply_chat_template(\n                messages,\n                tokenize=False,\n                add_generation_prompt=True,\n                enable_thinking=False,\n            )\n        except TypeError:\n            return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n    return prompt_text\n\n\ndef _empty_review_fields() -> dict[str, str]:\n    return {\n        "task_accuracy_0_to_3": "",\n        "instruction_adherence_0_to_2": "",\n        "completeness_0_to_2": "",\n        "clarity_0_to_2": "",\n        "no_unsupported_details_0_to_1": "",\n        "self_evaluation_score_1_to_10": "",\n        "review_notes": "",\n    }\n\n\ndef run_matrix(\n    models: list[dict],\n    output_path: Path,\n    device_request: str = "auto",\n    dtype_request: str = "auto",\n    max_new_tokens: int = 220,\n    resume: bool = True,\n) -> list[dict]:\n    """Generate and checkpoint one output for every selected model/prompt pair."""\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    validate_contract()\n    rows = _read_rows(output_path) if resume else []\n    # Failed rows remain visible after a run, but a later resume retries them.\n    rows = [row for row in rows if row.get("status") == "ok"]\n    completed = {(row["model_id"], row["prompt_id"]) for row in rows}\n    device = _device_name(device_request, torch)\n\n    for model_info in models:\n        model_id = model_info["model_id"]\n        remaining = [prompt for prompt in PROMPTS if (model_id, prompt["prompt_id"]) not in completed]\n        if not remaining:\n            continue\n\n        tokenizer = model = None\n        try:\n            tokenizer = AutoTokenizer.from_pretrained(model_id)\n            if tokenizer.pad_token_id is None:\n                tokenizer.pad_token = tokenizer.eos_token\n            if dtype_request == "float16":\n                dtype = torch.float16\n            elif dtype_request == "float32":\n                dtype = torch.float32\n            else:\n                dtype = torch.float16 if device in {"cuda", "mps"} else torch.float32\n            model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype, low_cpu_mem_usage=True)\n            model.to(device)\n            model.eval()\n\n            for prompt in remaining:\n                prompt_text = render_prompt(prompt)\n                formatted = _format_input(tokenizer, prompt_text)\n                inputs = tokenizer(formatted, return_tensors="pt").to(device)\n                started = time.perf_counter()\n                with torch.inference_mode():\n                    output = model.generate(\n                        **inputs,\n                        max_new_tokens=max_new_tokens,\n                        do_sample=False,\n                        pad_token_id=tokenizer.pad_token_id,\n                        eos_token_id=tokenizer.eos_token_id,\n                    )\n                _synchronize(device, torch)\n                latency = time.perf_counter() - started\n                generated_ids = output[0, inputs["input_ids"].shape[1] :]\n                output_tokens = int(generated_ids.shape[0])\n                generated = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()\n                row = {\n                    "model_id": model_id,\n                    "model_category": model_info["category"],\n                    "owner": model_info["owner"],\n                    "prompt_id": prompt["prompt_id"],\n                    "prompt_type": prompt["prompt_type"],\n                    "prompt": prompt["input"],\n                    "latency_seconds": round(latency, 4),\n                    "output_tokens": output_tokens,\n                    "tokens_per_second": round(output_tokens / latency, 4) if latency else "",\n                    "generated_output": generated,\n                    "status": "ok",\n                    "error": "",\n                    "device": device,\n                    "max_new_tokens": max_new_tokens,\n                    "generated_at_utc": datetime.now(timezone.utc).isoformat(),\n                    **_empty_review_fields(),\n                }\n                rows.append(row)\n                completed.add((model_id, prompt["prompt_id"]))\n                _write_rows(output_path, rows)\n        except Exception as exc:\n            error = f"{type(exc).__name__}: {str(exc)[:500]}"\n            for prompt in remaining:\n                key = (model_id, prompt["prompt_id"])\n                if key in completed:\n                    continue\n                rows.append(\n                    {\n                        "model_id": model_id,\n                        "model_category": model_info["category"],\n                        "owner": model_info["owner"],\n                        "prompt_id": prompt["prompt_id"],\n                        "prompt_type": prompt["prompt_type"],\n                        "prompt": prompt["input"],\n                        "latency_seconds": "",\n                        "output_tokens": "",\n                        "tokens_per_second": "",\n                        "generated_output": "",\n                        "status": "error",\n                        "error": error,\n                        "device": device,\n                        "max_new_tokens": max_new_tokens,\n                        "generated_at_utc": datetime.now(timezone.utc).isoformat(),\n                        **_empty_review_fields(),\n                    }\n                )\n                completed.add(key)\n            _write_rows(output_path, rows)\n        finally:\n            del model, tokenizer\n            gc.collect()\n            if device == "cuda":\n                torch.cuda.empty_cache()\n            elif device == "mps":\n                torch.mps.empty_cache()\n\n    validate_result_rows(rows, models, PROMPTS)\n    return rows\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    selection = parser.add_mutually_exclusive_group()\n    selection.add_argument("--owner", choices=["P1", "P2", "P3", "architecture"], default="P1")\n    selection.add_argument("--models", nargs="+")\n    parser.add_argument("--output", type=Path, default=Path("a7/results/person1_benchmark.csv"))\n    parser.add_argument("--device", choices=["auto", "cpu", "cuda", "mps"], default="auto")\n    parser.add_argument("--dtype", choices=["auto", "float16", "float32"], default="auto")\n    parser.add_argument("--max-new-tokens", type=int, default=220)\n    parser.add_argument("--restart", action="store_true", help="Replace rather than resume an existing output CSV")\n    parser.add_argument("--dry-run", action="store_true")\n    args = parser.parse_args()\n\n    models = select_models(model_ids=args.models) if args.models else select_models(owner=args.owner)\n    if args.dry_run:\n        print(json.dumps({"models": models, "prompts": len(PROMPTS), "evaluations": len(models) * len(PROMPTS)}, indent=2))\n        return\n    rows = run_matrix(\n        models,\n        args.output,\n        device_request=args.device,\n        dtype_request=args.dtype,\n        max_new_tokens=args.max_new_tokens,\n        resume=not args.restart,\n    )\n    print(f"Saved {len(rows)} rows to {args.output}")\n\n\nif __name__ == "__main__":\n    main()\n', 'review_results.py': '"""Apply the documented A7 self-evaluation rubric to generated outputs."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nimport re\nfrom pathlib import Path\n\nfrom a7.benchmark_contract import PROMPTS\nfrom a7.benchmark_runner import RESULT_COLUMNS\n\n\nEXPECTED_LABELS = {\n    "classification-01": "SHOPPING",\n    "classification-02": "MAINTENANCE",\n    "classification-03": "APPOINTMENT",\n    "classification-04": "DEADLINE",\n    "classification-05": "CHORE",\n    "priority-01": "HIGH",\n    "priority-02": "LOW",\n    "priority-03": "HIGH",\n    "priority-04": "URGENT",\n    "priority-05": "MEDIUM",\n}\n\nEXPECTED_CONCEPTS = {\n    "extraction-01": [("clean", "kitchen"), ("buy", "snack"), ("fix", "tap")],\n    "extraction-02": [("dentist",), ("smoke", "batter"), ("library", "book"), ("guest", "sheet")],\n    "extraction-03": [("vacuum",), ("cake",), ("stair", "rail")],\n    "extraction-04": [("electricity", "bill"), ("vet",), ("detergent",)],\n    "extraction-05": [("refrigerator", "noise"), ("closet",), ("tax", "document")],\n    "summary-01": [("old", "food"), ("shelf",), ("organiz",)],\n    "summary-02": [("clinic",), ("after 3",), ("calendar",)],\n    "summary-03": [("three", "month"), ("provider",), ("increase",)],\n    "summary-04": [("book",), ("due",), ("weekend",)],\n    "summary-05": [("sheet",), ("towel",), ("bin",), ("lamp",)],\n    "planning-01": [("clean",), ("bed", "linen", "sleep"), ("toiletr", "towel", "suppl")],\n    "planning-02": [("list", "meal"), ("price", "sale", "budget"), ("90", "cost", "total")],\n    "planning-03": [("suppl", "prepare"), ("surface", "counter"), ("floor", "mop"), ("safe", "sanit")],\n    "planning-04": [("declutter", "sort"), ("pack",), ("label",), ("move", "date", "transport")],\n    "planning-05": [("light",), ("unplug", "device"), ("thermostat", "heat", "cool"), ("monitor", "usage")],\n}\n\n\ndef _json_value(output: str):\n    text = output.strip()\n    text = re.sub(r"^```(?:json)?\\s*", "", text, flags=re.I)\n    text = re.sub(r"\\s*```$", "", text)\n    try:\n        return json.loads(text)\n    except (json.JSONDecodeError, TypeError):\n        return None\n\n\ndef _concept_coverage(prompt_id: str, output: str) -> tuple[int, int]:\n    lowered = output.lower()\n    concepts = EXPECTED_CONCEPTS[prompt_id]\n    matched = sum(any(term in lowered for term in alternatives) for alternatives in concepts)\n    return matched, len(concepts)\n\n\ndef _is_repetitive(output: str) -> bool:\n    words = re.findall(r"[a-z]+", output.lower())\n    if len(words) < 12:\n        return False\n    unique_ratio = len(set(words)) / len(words)\n    return unique_ratio < 0.22\n\n\ndef review_output(prompt_id: str, output: str) -> dict[str, int | str]:\n    """Score one output conservatively using the shared ten-point rubric."""\n    prompt_type = next(item["prompt_type"] for item in PROMPTS if item["prompt_id"] == prompt_id)\n    text = output.strip()\n    parsed = _json_value(text)\n    accuracy = adherence = completeness = clarity = supported = 0\n    notes: list[str] = []\n\n    if prompt_type in {"classification", "priority_prediction"}:\n        expected = EXPECTED_LABELS[prompt_id]\n        exact = text.upper().rstrip(".") == expected\n        first_label = re.match(r"^([A-Z]+)", text.upper())\n        correct = bool(first_label and first_label.group(1) == expected)\n        accuracy = 3 if correct else 0\n        adherence = 2 if exact else (1 if correct else 0)\n        completeness = 2 if correct else 0\n        clarity = 2 if exact else (1 if correct else 0)\n        supported = 1 if correct else 0\n        result = "exactly matched" if exact else "correct label with extra text" if correct else "did not match"\n        notes.append(f"Expected exact label {expected}; {result}.")\n    else:\n        matched, total = _concept_coverage(prompt_id, text)\n        ratio = matched / total\n        accuracy = 3 if ratio >= 0.8 else 2 if ratio >= 0.5 else 1 if matched else 0\n        completeness = 2 if ratio == 1 else 1 if matched else 0\n\n        if prompt_type == "structured_extraction":\n            valid = isinstance(parsed, dict) and isinstance(parsed.get("tasks"), list)\n            expected_count = len(EXPECTED_CONCEPTS[prompt_id])\n            schema_ok = valid and len(parsed["tasks"]) == expected_count and all(\n                isinstance(task, dict)\n                and {"title", "task_type", "priority", "description"} <= set(task)\n                for task in parsed["tasks"]\n            )\n            adherence = 2 if schema_ok else 0\n            notes.append(f"Covered {matched}/{total} concerns; JSON task schema {\'valid\' if schema_ok else \'invalid\'}.")\n        elif prompt_type == "summarization":\n            schema_ok = isinstance(parsed, dict) and set(parsed) == {"title", "summary"}\n            adherence = 2 if schema_ok else 0\n            notes.append(f"Covered {matched}/{total} details; summary JSON {\'valid\' if schema_ok else \'invalid\'}.")\n        else:\n            steps = parsed.get("steps") if isinstance(parsed, dict) else None\n            schema_ok = isinstance(steps, list) and 3 <= len(steps) <= 5 and all(isinstance(step, str) for step in steps)\n            adherence = 2 if schema_ok else 0\n            notes.append(f"Covered {matched}/{total} planning concepts; 3–5 step JSON {\'valid\' if schema_ok else \'invalid\'}.")\n\n        repetitive = _is_repetitive(text)\n        clarity = 2 if schema_ok and not repetitive else 1 if text and matched and not repetitive else 0\n        unsupported_markers = (\n            "i have completed", "i have successfully", "confirmed appointment",\n            "next wednesday", "i will send", "your dietary",\n        )\n        invented = any(marker in text.lower() for marker in unsupported_markers)\n        supported = 0 if invented else (1 if matched else 0)\n        if repetitive:\n            notes.append("Output is repetitive.")\n        if invented:\n            notes.append("Output adds an unsupported completion claim or specific detail.")\n\n    total_score = accuracy + adherence + completeness + clarity + supported\n    return {\n        "task_accuracy_0_to_3": accuracy,\n        "instruction_adherence_0_to_2": adherence,\n        "completeness_0_to_2": completeness,\n        "clarity_0_to_2": clarity,\n        "no_unsupported_details_0_to_1": supported,\n        "self_evaluation_score_1_to_10": max(1, total_score),\n        "review_notes": " ".join(notes),\n    }\n\n\ndef review_csv(path: Path) -> None:\n    with path.open(newline="", encoding="utf-8") as handle:\n        rows = list(csv.DictReader(handle))\n    for row in rows:\n        if row["status"] == "ok":\n            row.update(review_output(row["prompt_id"], row["generated_output"]))\n        else:\n            row.update({\n                "task_accuracy_0_to_3": 0,\n                "instruction_adherence_0_to_2": 0,\n                "completeness_0_to_2": 0,\n                "clarity_0_to_2": 0,\n                "no_unsupported_details_0_to_1": 0,\n                "self_evaluation_score_1_to_10": 1,\n                "review_notes": f"Generation failed: {row[\'error\']}",\n            })\n    temporary = path.with_suffix(path.suffix + ".tmp")\n    with temporary.open("w", newline="", encoding="utf-8") as handle:\n        writer = csv.DictWriter(handle, fieldnames=RESULT_COLUMNS)\n        writer.writeheader()\n        writer.writerows(rows)\n    temporary.replace(path)\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("path", type=Path)\n    args = parser.parse_args()\n    review_csv(args.path)\n    print(f"Reviewed {args.path}")\n\n\nif __name__ == "__main__":\n    main()\n'}
HASHES = {'__init__.py': 'beb14e51bd4722c5a569637b9bf2930e1c67fe4ffeaa32c9ba5613cdc8ca5599', 'benchmark_contract.py': '87d109d66ecc713e7692e98f1da5dda1e354e18324ead006304aa6159ab1df4d', 'benchmark_runner.py': 'bae9e4be09584625ccb6849d04c089d41f00f1ece851d545adeb9aaef145a93b', 'review_results.py': 'd7f20a79b6927631e865e9e1907d3603acd5be25670215e4ac5380bcd40aa1d0'}
(WORK/'a7').mkdir(exist_ok=True)
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == HASHES[name]
    (WORK/'a7'/name).write_text(source, encoding='utf-8')
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
print('Shared benchmark source hashes verified. Ready for CUDA benchmark.')


In [ ]:
import torch, platform, importlib.metadata
from datetime import datetime, timezone
from a7.benchmark_runner import run_matrix, select_models
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
OUT = WORK/'results/architecture_tinyllama_cuda.csv'
OUT.parent.mkdir(parents=True, exist_ok=True)
ENV = dict(recorded_at_utc=datetime.now(timezone.utc).isoformat(), platform=platform.platform(),
    device='cuda', dtype='float16', gpu=torch.cuda.get_device_name(0),
    vram_bytes=torch.cuda.get_device_properties(0).total_memory,
    max_new_tokens=220, do_sample=False, contract_source_sha256=HASHES,
    packages={p:importlib.metadata.version(p) for p in ['torch','transformers','huggingface-hub']},
    timing_scope='Unchanged shared runner: prefill plus decoding, excludes model loading; no warmup')
OUT.with_suffix('.environment.json').write_text(json.dumps(ENV,indent=2),encoding='utf-8')
print('GPU:',ENV['gpu'],flush=True)
rows = run_matrix(select_models(model_ids=['TinyLlama/TinyLlama-1.1B-Chat-v1.0']),
    OUT, device_request='cuda', dtype_request='float16')
print('TinyLlama successful evaluations:',sum(r['status']=='ok' for r in rows),'/ 25')


In [ ]:
from a7.review_results import review_csv
import csv, zipfile
review_csv(OUT)
with OUT.open(encoding='utf-8',newline='') as handle:
    results = list(csv.DictReader(handle))
print('Successful:',sum(r['status']=='ok' for r in results),'/ 25')
print('Automated rubric scores require inspection against actual responses.')
BUNDLE = WORK/'architecture_tinyllama_cuda_results.zip'
with zipfile.ZipFile(BUNDLE,'w',zipfile.ZIP_DEFLATED) as z:
    z.write(OUT,OUT.name)
    z.write(OUT.with_suffix('.environment.json'),OUT.with_suffix('.environment.json').name)
print('Results bundle:',BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
